In [2]:
import sys
!{sys.executable} -m pip install duckdb

In [1]:
import duckdb
import json

In [2]:
files=["likes.parquet", "reposts.parquet", "list_blocks.parquet"]

values=[]

query="""
    SELECT distinct subject_collection AS collection
    FROM read_parquet('{file}')
"""

for file in files:
    result = duckdb.query(query.format(file=file))

    cols = [c[0] for c in result.description]

    rows = result.fetchall()
    
    for row in rows:
        if row[0] not in values:
            values.append(row[0])



query="""
    SELECT distinct list_collection AS collection
    FROM read_parquet('{file}')
"""
result = duckdb.query(query.format(file="list_items.parquet"))

cols = [c[0] for c in result.description]

rows = result.fetchall()
#print(rows)

for row in rows:
    if row[0] not in values:
        values.append(row[0])


print(values)

##   LIST_BLOCK ONLY HAS COLLECTION 'app.bsky.graph.list'


IOException: IO Error: No files found that match the pattern "likes.parquet"

In [4]:
print(len(values))

16


In [15]:
files=["posts.parquet", "profiles.parquet", "lists.parquet"]

query="""
    SELECT labels AS labels
    FROM read_parquet('{file}')
"""

labels=[]
count=0

for file in files:
    result = duckdb.query(query.format(file=file))

    cols = [c[0] for c in result.description]

    rows = result.fetchall()
    
    for row in rows:
        if row[0] != None:
            for el in json.loads(row[0])["values"]:
                found = False

                for item in labels:
                    if item["val"] == el["val"]:
                        item["freq"] += 1
                        found = True
                        break 

                if not found:
                    labels.append({"val": el["val"], "freq": 1})


print(sorted(labels, key=lambda dic: dic["freq"], reverse=True))


MemoryError: 

In [16]:
import duckdb
import json
import collections

# --- Configuration ---
files = ["posts.parquet", "profiles.parquet", "lists.parquet"]
query = """
    SELECT labels AS labels
    FROM read_parquet('{file}')
"""
# Define a chunk size. 100,000 is a good starting point, adjust based on 
# your system's available memory and the size of your rows.
CHUNK_SIZE = 100000 

# Use a dictionary (or collections.defaultdict) for efficient frequency counting.
# Key: label value (el["val"])
# Value: frequency count (int)
label_freqs = collections.defaultdict(int)

# --- Memory-Efficient Processing Loop ---
for file in files:
    print(f"Processing file: {file}")
    
    # 1. Execute the query
    result = duckdb.query(query.format(file=file))

    # We don't need cols or rows list anymore, we process chunks iteratively.
    
    # 2. Iterate in chunks to avoid MemoryError
    while True:
        # Fetch a chunk of data
        chunk = result.fetchmany(CHUNK_SIZE)
        
        # Stop if the chunk is empty (end of results)
        if not chunk:
            break
            
        # 3. Process the chunk
        for row in chunk:
            # Check for NULL labels column
            if row[0] is not None:
                try:
                    # Parse the JSON string
                    data = json.loads(row[0])
                    
                    # Ensure the structure is as expected and iterate over values
                    if "values" in data and isinstance(data["values"], list):
                        for el in data["values"]:
                            # Use the dictionary for O(1) frequency counting
                            # This replaces the entire slow O(N) linear search loop
                            if "val" in el:
                                label_freqs[el["val"]] += 1
                                
                except json.JSONDecodeError:
                    # Handle cases where the label string is not valid JSON
                    print(f"Warning: Invalid JSON found in file {file}.")
                except Exception as e:
                    # Handle other potential errors (e.g., missing 'values' or 'val' keys)
                    pass


# --- Final Output Formatting ---

# 4. Convert the efficient dictionary back to the required list of dictionaries format
#    (list of {"val": ..., "freq": ...})
labels = [{"val": val, "freq": freq} for val, freq in label_freqs.items()]

# 5. Print the sorted results
print("\n--- Top Labels ---")
print(sorted(labels, key=lambda dic: dic["freq"], reverse=True))

Processing file: posts.parquet
Processing file: profiles.parquet
Processing file: lists.parquet

--- Top Labels ---
[{'val': 'porn', 'freq': 1473547}, {'val': 'sexual', 'freq': 588688}, {'val': 'nudity', 'freq': 382741}, {'val': '!no-unauthenticated', 'freq': 250914}, {'val': 'graphic-media', 'freq': 128895}, {'val': '!warn', 'freq': 259}, {'val': 'spoiler', 'freq': 120}, {'val': 'gore', 'freq': 76}, {'val': 'nsfw', 'freq': 28}, {'val': 'graysky.app', 'freq': 23}, {'val': 'circle', 'freq': 23}, {'val': 'Nudity', 'freq': 15}, {'val': '!hide', 'freq': 13}, {'val': 'Nsfw', 'freq': 13}, {'val': 'nsfl', 'freq': 11}, {'val': 'graphic', 'freq': 10}, {'val': 'uspol', 'freq': 9}, {'val': 'Food', 'freq': 8}, {'val': 'The Saturday Paper Quiz', 'freq': 7}, {'val': 'bridged-from-bridgy-fed-activitypub', 'freq': 7}, {'val': 'Selfie', 'freq': 6}, {'val': 'azsky-community-post', 'freq': 5}, {'val': 'Uspol', 'freq': 4}, {'val': 'Silly', 'freq': 3}, {'val': 'Silly Cat', 'freq': 3}, {'val': '触手', 'freq':

In [7]:
di=sorted(labels, key=lambda dic: dic["freq"], reverse=True)

tot=0
for el in di:
    tot+=el["freq"]

su=0
for i in range(16):
    su+=di[i]["freq"]
    

for i in range(16):
    print(di[i]["val"])

print(su, tot)
print(su/tot)

NameError: name 'labels' is not defined

In [4]:
languages=[]
query="""
    SELECT distinct languages AS languages
    FROM read_parquet('posts.parquet')
"""
result = duckdb.query(query)

cols = [c[0] for c in result.description]

rows = result.fetchall()

for row in rows:
    if len(json.loads(row[0])) != 0:
        if json.loads(row[0])[0] not in languages:
            languages.append(json.loads(row[0])[0])

print(languages)


# one hot encoding for cultural regions or binary encoding for all

['pt', 'cs', 'sk', 'en', 'eu', 'no', 'yi', 'es', 'tr', 'fi', 'ga', 'id', 'ro', 'uk', 'ca', 'pl', 'de', 'yue', 'af', 'nn', 'fr', 'ber', 'ja', 'nl', 'iw', 'ar', 'ak', 'jbo', 'sv', 'vi', 'hu', 'br', 'an', 'aa', 'tl', 'hi', 'th', 'ab', 'ast', 'tlh', 'ko', 'ff', 'Angika', 'bs', 'my', 'ru', 'am', 'la', 'nb', 'zh', 'as', 'is', 'el', 'rn', 'kn', 'en-US', 'fy', 'vo', 'lv', 'sr', 'en-NZ', 'it', 'kk', 'en-us', 'lt', 'eo', 'et', 'lo', 'tk', 'fa', 'da', 'chr', 'bm', 'UK', 'fr-CA', 'mk', 'hr', 'gsw-u-sd-chzh', 'oj', 'ba', 'pt-Br', 'gv', 'po', 'bg', 'av', 'ja-JP', 'gsw', 'es-ES', 'en-AU', 'cy', 'EN', 'dz', 'brx', 'moh', 'lu', 'ay', 'az', 'be', 'mn', 'km', 'PT-PT', 'ae', 'sq', 'cv', 'gd', 'in', 'he', 'hy', 'lad', 'sw', 'cr', 'ars', 'uk-UA', 'ie', 'zh-CN', 'brazil', 'vi-VN', 'hu-HU', 'bn', 'hsb', 'ms', 'fo', 'oc', 'IT', 'mg', 'ml', 'ee', 'MY', 'und', 'gl', 'se', 'scn', 'ca-ES', 'en-ZA', 'bo', 'io', 'sl', 'ig', 'ksh', 'haw', 'ku', 'nv', 'tt', 'ceb', 'cu', 'lb', 'en-GB', 'PL', 'co', 'ce', 'FR', 'ko-KR', 

In [5]:
lang_freq={}
query="""
    SELECT distinct languages AS languages
    FROM read_parquet('posts.parquet')
"""
CHUNK_SIZE = 100000 


result = duckdb.query(query)

while True:
    chunk = result.fetchmany(CHUNK_SIZE)
    
    # Stop if the chunk is empty (end of results)
    if not chunk:
        break
        
    # 3. Process the chunk
    for row in chunk:

        # Check for NULL labels column
        if row[0] is not None:
            try:
                # Parse the JSON string
                lang=json.loads(row[0])
                
                # Ensure the structure is as expected and iterate over values
                
                for el in lang:
                    
                    if el not in lang_freq.keys():
                        lang_freq[el] = 1

                    else:
                        lang_freq[el] += 1

            except json.JSONDecodeError:
                # Handle cases where the label string is not valid JSON
                print(f"Warning: Invalid JSON found in file {file}.")
            except Exception as e:
                # Handle other potential errors (e.g., missing 'values' or 'val' keys)
                pass


sorted_items = sorted(lang_freq.items(), key=lambda item: item[1], reverse=True)

# Since Python 3.7+, dictionaries maintain insertion order, 
# so you can convert the sorted list of tuples back into an ordered dictionary.
sorted_data = dict(sorted_items)

print(sorted_data)

{'en': 4326, 'pt': 2764, 'ja': 2127, 'ber': 1596, 'fr': 1583, 'es': 1544, 'nl': 1384, 'de': 1381, 'tlh': 1289, 'la': 1262, 'fi': 1162, 'rn': 1108, 'it': 1094, 'id': 1027, 'ro': 1015, 'hu': 956, 'ga': 934, 'af': 925, 'pl': 922, 'cs': 909, 'tl': 884, 'eo': 846, 'lt': 809, 'is': 808, 'et': 791, 'sv': 766, 'da': 751, 'no': 738, 'tr': 733, 'sk': 717, 'lv': 644, 'tk': 629, 'sr': 560, 'vo': 554, 'zh': 457, 'ko': 397, 'hi': 376, 'el': 314, 'ru': 300, 'ar': 223, 'uk': 219, 'vi': 150, 'am': 145, 'sq': 123, 'nb': 108, 'hy': 106, 'ak': 97, 'an': 96, 'fa': 96, 'he': 94, 'th': 88, 'aa': 84, 'yi': 84, 'kk': 80, 'ab': 79, 'Angika': 77, 'as': 76, 'ca': 67, 'av': 65, 'bg': 60, 'be': 54, 'hr': 53, 'ae': 48, 'cy': 43, 'mn': 42, 'az': 39, 'gl': 35, 'bs': 33, 'bn': 33, 'chr': 32, 'nn': 32, 'ay': 30, 'mk': 30, 'gd': 28, 'eu': 25, 'iw': 24, 'fil': 24, 'sl': 24, 'haw': 22, 'br': 21, 'in': 21, 'ms': 21, 'und': 20, 'ur': 20, 'mi': 19, 'ka': 19, 'yue': 18, 'kn': 16, 'ta': 16, 'ba': 15, 'bo': 15, 'io': 13, 'com': 

In [6]:
tot=0
su=0
for el in sorted_data.values():
    tot+=el

freq=list(sorted_data.values())
lan=list(sorted_data.keys())

for i in range(40):
    su+=freq[i]


print(lan[:40])
    

print(su, tot)
print(su/tot)

['en', 'pt', 'ja', 'ber', 'fr', 'es', 'nl', 'de', 'tlh', 'la', 'fi', 'rn', 'it', 'id', 'ro', 'hu', 'ga', 'af', 'pl', 'cs', 'tl', 'eo', 'lt', 'is', 'et', 'sv', 'da', 'no', 'tr', 'sk', 'lv', 'tk', 'sr', 'vo', 'zh', 'ko', 'hi', 'el', 'ru', 'ar']
41605 45187
0.9207294133268418


In [11]:
query="""
    SELECT purpose AS purpose
    FROM read_parquet('lists.parquet')
"""
CHUNK_SIZE = 100000 

purpose_freq={}


result = duckdb.query(query)

while True:
    chunk = result.fetchmany(CHUNK_SIZE)
    
    # Stop if the chunk is empty (end of results)
    if not chunk:
        break
        
    # 3. Process the chunk
    for row in chunk:
        
        # Check for NULL labels column
        if row[0] is not None:
            
            if row[0] not in purpose_freq.keys():
                purpose_freq[row[0]] = 1

            else:
                purpose_freq[row[0]] += 1

           


sorted_items = sorted(purpose_freq.items(), key=lambda item: item[1], reverse=True)

# Since Python 3.7+, dictionaries maintain insertion order, 
# so you can convert the sorted list of tuples back into an ordered dictionary.
sorted_data = dict(sorted_items)

print(sorted_data)

{'app.bsky.graph.defs#curatelist': 197784, 'app.bsky.graph.defs#referencelist': 116356, 'app.bsky.graph.defs#modlist': 19310, 'app.bsky.graph.defs#curateList': 3, 'app.bsky.graph.list#modlist': 1, 'purpose': 1, 'Mostrar quem esta me seguindo': 1, 'spammers': 1, 'blocked': 1, 'app.bsky.graph.top8': 1, 'app.bsky.graph.defs#curationlist': 1}


In [18]:
import ast

query="""
    SELECT languages AS languages
    FROM read_parquet('datasets\posts.parquet')
"""
result = duckdb.query(query)

chunk = result.fetchmany(100000)

print(ast.literal_eval(chunk[0][0]))

for el in chunk:
    if len(ast.literal_eval(el[0])) != 1:
        print(el)

['en']
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('["ar", "en"]',)
('[

## EVENT STRUCTURE

event: follows, blocks
structure: event_type, created_at, 0, 0, 0, 0

event: likes, reposts, list_blocks, list_item
structure: event_type, created_at, collection, 0, 0, 0

event: lists
structure: event_type, created_at, collection, labels, purpose, 0

event: profiles
structure: event_type, created_at, 0, labels, 0, 0

event: post
structure: event_type, created_at, 0, labels, 0, languages




## 22 EVENT TYPES (A-B):

like, follow, repost, list_block, block where did_id=A and subject_id=B
like, follow, repost, list_block, block where did_id=B and subject_id=A

list, profiles where did_id=A
list, profiles where did_id=B

post where did_id=A and root_id=B
post where did_id=A and parent_id=B
post where did_id=B and root_id=A
post where did_id=B and parent_id=A

list_item where did_id=A and list_creator=B
list_item where did_id=A and subject_id=B
list_item where did_id=B and list_creator=A
list_item where did_id=B and subject_id=A
